# Film Değerlendirmeleri ve Kelime Torbası Modellemesi

🎯 Bu meydan okumanın amacı metinlerin ***Kelime Torbası*** modellemesi ile oynamaktır.

✍️ Aşağıdaki veri setinde, "olumlu" veya "olumsuz" olarak sınıflandırılmış $2000$ değerlendirme bulunmaktadır.

In [ ]:
import pandas as pd

data = pd.read_csv("https://d32aokrjazspmn.cloudfront.net/materials/movie_reviews.csv")
data.head()

,target,reviews
0,neg,"plot : two teen couples go to a church party ,..."
1,neg,the happy bastard's quick movie review \ndamn ...
2,neg,it is movies like these that make a jaded movi...
3,neg,""" quest for camelot "" is warner bros . ' firs..."
4,neg,synopsis : a mentally unstable man undergoing ...


In [ ]:
data.shape

(2000, 2)

## 1. Ön İşleme

❓ **Soru (Metin Temizleme)** ❓

- Bir cümleyi temizleyecek `preprocessing` fonksiyonu yazın ve tüm değerlendirmelerimize uygulayın. Bu fonksiyon şunları yapmalı:
    - boşlukları kaldırmak
    - karakterleri küçük harfe çevirmek
    - sayıları kaldırmak
    - noktalama işaretlerini kaldırmak
    - kelime parçacıklarına ayırmak (tokenize)
    - kök haline getirmek (lemmatize)
- Temizlenmiş değerlendirmeleri `clean_reviews` adında bir sütunda saklayabilirsiniz.
- Bu meydan okumada durak kelimeleri kaldırmayın, nedenini `3. N-gram modelleme` bölümünde açıklayacağız

In [ ]:
import string
from nltk import word_tokenize
from nltk.stem import WordNetLemmatizer

def preprocessing(sentence):
    # $CHALLENGIFY_BEGIN

    # Removing whitespaces
    sentence = sentence.strip()
    # Lowercasing
    sentence = sentence.lower()
    # Removing numbers
    sentence = ''.join(char for char in sentence if not char.isdigit())
    # Removing punctuation
    for punctuation in string.punctuation:
        sentence = sentence.replace(punctuation, '')
    # Tokenizing
    tokenized = word_tokenize(sentence)
    # Lemmatizing
    lemmatizer = WordNetLemmatizer()
    lemmatized = [lemmatizer.lemmatize(word) for word in tokenized]
    cleaned_sentence = " ".join(lemmatized)
    return cleaned_sentence

    # $CHALLENGIFY_END


In [ ]:
# Clean reviews
# $CHALLENGIFY_BEGIN
data['clean_reviews'] = data.reviews.apply(preprocessing)
data.head()
# $CHALLENGIFY_END

,target,reviews,clean_reviews
0,neg,"plot : two teen couples go to a church party ,...",plot two teen couple go to a church party drin...
1,neg,the happy bastard's quick movie review \ndamn ...,the happy bastard quick movie review damn that...
2,neg,it is movies like these that make a jaded movi...,it is movie like these that make a jaded movie...
3,neg,""" quest for camelot "" is warner bros . ' firs...",quest for camelot is warner bros first feature...
4,neg,synopsis : a mentally unstable man undergoing ...,synopsis a mentally unstable man undergoing ps...


❓ **Soru (Etiket Kodlaması)**❓

Hedefinizi EtiketKodlayın ve `"target_encoded"` adlı bir sütunda saklayın

In [ ]:
from sklearn import preprocessing

le = preprocessing.LabelEncoder()
data["target_encoded"] =  le.fit_transform(data.target)

In [ ]:
# Quick check
data.head()

,target,reviews,clean_reviews,target_encoded
0,neg,"plot : two teen couples go to a church party ,...",plot two teen couple go to a church party drin...,0
1,neg,the happy bastard's quick movie review \ndamn ...,the happy bastard quick movie review damn that...,0
2,neg,it is movies like these that make a jaded movi...,it is movie like these that make a jaded movie...,0
3,neg,""" quest for camelot "" is warner bros . ' firs...",quest for camelot is warner bros first feature...,0
4,neg,synopsis : a mentally unstable man undergoing ...,synopsis a mentally unstable man undergoing ps...,0


## 2. Kelime Torbası Modellemesi

❓ **Soru (Tekli kelimelerle NaiveBayes)** ❓

`cross_validate` kullanarak, metinlerin Kelime Torbası temsili üzerinde eğitilmiş bir Çok Terimli Naive Bayes modelini puanlayın.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import cross_validate

vectorizer = CountVectorizer()
naivebayes = MultinomialNB()

X_bow = vectorizer.fit_transform(data.clean_reviews)

cv_nb = cross_validate(
    naivebayes,
    X_bow,
    data.target_encoded,
    scoring = "accuracy"
)

round(cv_nb['test_score'].mean(),2)

0.82

## 3. N-gram Modellemesi

👀 Durak kelimeleri kaldırmamanızı istediğimizi hatırlayın. Neden? 

👉 Naive Bayes modelini ikili kelimelerle (bigram) eğiteceğiz. Dolayısıyla "Kişnişi sevmiyorum" gibi bir cümlede, örneğin bu cümledeki olumsuzluğu tespit etmek için "sevmi yorum" ikili kelimesini taramak önemlidir.

❓ **Soru (İkili kelimelerle NaiveBayes)** ❓

`cross_validate` kullanarak, metinlerin 2-gram Kelime Torbası temsili üzerinde eğitilmiş bir Çok Terimli Naive Bayes modelini puanlayın.

In [ ]:
vectorizer = CountVectorizer(ngram_range = (2,2))
naivebayes = MultinomialNB()

X_bow = vectorizer.fit_transform(data.clean_reviews)

cv_nb = cross_validate(
    naivebayes,
    X_bow,
    data.target_encoded,
    scoring = "accuracy"
)

round(cv_nb['test_score'].mean(),2)

0.84

🏁 Tebrikler! Artık vektörize edilmiş metinler üzerinde bir Naive Bayes modelini nasıl eğiteceğinizi biliyorsunuz.

💾 Notebook'unuzu `git add/commit/push` yapmayı unutmayın...

🚀 ... ve bir sonraki meydan okumaya geçin!